# HyDE [Step 2 - Hypothetical Document Embeddings]

> **MLCourse - Agentic AI - Advanced RAG - Query Transformation**

**HyDE** (Hypothetical Document Embeddings, Gao et al. 2022) is one of those
ideas that sounds wrong until you see the numbers.

The recipe:

1. Ask an LLM to **answer the question** - with no retrieval, from its own
   parametric memory.
2. Throw the answer's *facts* away. You do not trust them at all.
3. **Embed that hypothetical answer** and use its vector as the search query.
4. Retrieve, then generate the real answer from the retrieved documents.

The counter-intuitive part is step 2. HyDE does not care whether the
hypothetical answer is factually correct. It only needs the answer to be
**written in the register, vocabulary and shape of a real answer** - because
that is what makes it land near real answers in embedding space.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# Paragraph-sized chunks: human-readable units, good enough for retrieval demos
# and identical to the chunking used in ../01_hybrid_search.
paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("characters :", len(raw_text))
print("paragraphs :", len(paragraphs))
print("example    :", paragraphs[10][:150], "...")

characters : 144696
paragraphs : 237
example    : Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passa ...


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vectors = encoder.encode(paragraphs, normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)


def embed(text):
    return encoder.encode([text], normalize_embeddings=True)[0]


def dense_rank(text, top_n=10):
    """Rank paragraph indices by cosine similarity to `text` (best first)."""
    sims = doc_vectors @ embed(text)
    return [int(i) for i in np.argsort(sims)[::-1][:top_n]]


def dense_scores(text):
    return doc_vectors @ embed(text)


print("dense index ready:", doc_vectors.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

dense index ready: (237, 384)


### 2. Generating a hypothetical document

The prompt matters more than you would expect. Three rules:

- **Ask for a passage, not an answer.** "Write a paragraph from the book" beats
  "answer this question" - the former produces document-register text.
- **Constrain the length.** A 60-100 word passage matches typical chunk size.
  A three-word answer has too little signal; a 500-word essay drifts off-topic.
- **Let it be wrong.** Do not add "only answer if you are sure". Refusals and
  hedging ("I do not have enough information") are the worst possible HyDE
  output - they embed near other hedging text, which is nowhere near your
  documents.

In [5]:
HYDE_PROMPT = (
    "Write a short passage (about 70 words) that would plausibly appear in "
    "Lewis Carroll's 'Alice's Adventures in Wonderland' and that answers the "
    "question below. Write it in the narrative style of the novel. "
    "Do not hedge, do not mention that you are uncertain, and do not refer to "
    "the question - just write the passage.\n\n"
    "Question: {question}\n\nPassage:"
)


def hyde_document(question):
    return ask(HYDE_PROMPT.format(question=question))


QUESTION = "What advice does the Caterpillar give Alice?"
hypothetical = hyde_document(QUESTION)

print("QUESTION:", QUESTION)
print("\nHYPOTHETICAL DOCUMENT (generated, not retrieved, possibly wrong):")
print(hypothetical)

QUESTION: What advice does the Caterpillar give Alice?

HYPOTHETICAL DOCUMENT (generated, not retrieved, possibly wrong):
"Who are you?" said the Caterpillar.

This was not an encouraging opening for a conversation. Alice replied, rather shyly, "I—I hardly know, sir, just at present—at least I know who I was when I got up this morning, but I think I must have been changed several times since then."

"What do you mean by that?" said the Caterpillar sternly. "Explain yourself!"

"I can't explain myself, I'm afraid, sir," said Alice, "because I'm not myself, you see."


Notice what the model produced: narrative prose, past tense, character names,
concrete objects. Some of it may be invented - the Caterpillar may be given
advice he never gave. **That does not matter.** We are about to use this text
only as a *direction to search in*.

### 3. Search with the hypothetical document

Now the actual HyDE step: embed the passage instead of the question.

In [6]:
def hyde_retrieve(question, top_n=5, hypothetical_doc=None):
    """Embed a generated answer passage and search with that vector."""
    doc = hypothetical_doc or hyde_document(question)
    return dense_rank(doc, top_n=top_n), doc


baseline = dense_rank(QUESTION, top_n=5)
hyde_ids, _ = hyde_retrieve(QUESTION, top_n=5, hypothetical_doc=hypothetical)

print("BASELINE - embedding the raw question:")
for rank, doc_id in enumerate(baseline, 1):
    print(f"  #{rank} doc_{doc_id}: {paragraphs[doc_id][:100]}...")

print("\nHyDE - embedding the hypothetical passage:")
for rank, doc_id in enumerate(hyde_ids, 1):
    tag = "NEW" if doc_id not in baseline else f"was #{baseline.index(doc_id) + 1}"
    print(f"  #{rank} doc_{doc_id} ({tag}): {paragraphs[doc_id][:100]}...")

print(f"\noverlap between the two result sets: "
      f"{len(set(baseline) & set(hyde_ids))} of 5")

BASELINE - embedding the raw question:
  #1 doc_91: Which brought them back again to the beginning of the conversation. Alice felt a little irritated at...
  #2 doc_93: This time Alice waited patiently until it chose to speak again. In a minute or two the Caterpillar t...
  #3 doc_90: “Well, perhaps you haven’t found it so yet,” said Alice; “but when you have to turn into a chrysalis...
  #4 doc_86: “And yet what a dear little puppy it was!” said Alice, as she leant against a buttercup to rest hers...
  #5 doc_190: So Alice began telling them her adventures from the time when she first saw the White Rabbit. She wa...

HyDE - embedding the hypothetical passage:
  #1 doc_91 (was #1): Which brought them back again to the beginning of the conversation. Alice felt a little irritated at...
  #2 doc_89 (NEW): This was not an encouraging opening for a conversation. Alice replied, rather shyly, “I—I hardly kno...
  #3 doc_190 (was #5): So Alice began telling them her adventures from the time wh

### 4. Why it moves: look at the similarity scores

The clearest way to see HyDE working is to compare the *score* the top true
answer paragraph receives under each query vector.

In [7]:
import numpy as np

q_scores = dense_scores(QUESTION)
h_scores = dense_scores(hypothetical)

union = list(dict.fromkeys(list(baseline) + list(hyde_ids)))

print(f"{'doc':>8}{'cos(question)':>16}{'cos(hyde)':>12}{'change':>10}  preview")
print("-" * 100)
for doc_id in union:
    delta = h_scores[doc_id] - q_scores[doc_id]
    print(f"doc_{doc_id:<4}{q_scores[doc_id]:>16.3f}{h_scores[doc_id]:>12.3f}"
          f"{delta:>+10.3f}  {paragraphs[doc_id][:45]}...")

print(f"\nmean similarity to the whole corpus:")
print(f"  question vector    : {q_scores.mean():.3f}")
print(f"  hypothetical vector: {h_scores.mean():.3f}")

     doc   cos(question)   cos(hyde)    change  preview
----------------------------------------------------------------------------------------------------
doc_91             0.674       0.769    +0.095  Which brought them back again to the beginnin...
doc_93             0.642       0.493    -0.149  This time Alice waited patiently until it cho...
doc_90             0.588       0.518    -0.070  “Well, perhaps you haven’t found it so yet,” ...
doc_86             0.576       0.511    -0.065  “And yet what a dear little puppy it was!” sa...
doc_190            0.560       0.552    -0.008  So Alice began telling them her adventures fr...
doc_89             0.367       0.694    +0.327  This was not an encouraging opening for a con...
doc_101            0.526       0.536    +0.010  This was such a new idea to Alice, that she w...
doc_108            0.385       0.522    +0.137  “There might be some sense in your knocking,”...

mean similarity to the whole corpus:
  question vector    : 0.344


The hypothetical document sits in a *denser part of the corpus* - its average
similarity to everything is higher, because it is the same kind of text as the
corpus. That is HyDE's whole mechanism, and it is also its main risk: a
hypothetical document that drifts off-topic will confidently retrieve
off-topic documents.

### 5. Multi-sample HyDE

The original paper generates **several** hypothetical documents and averages
their embeddings. This is cheap insurance: one bad generation drags the mean
much less than it drags a single vector.

We use `temperature=0.7` here deliberately - we *want* variety between samples.

In [8]:
from langchain_groq import ChatGroq

varied_llm = ChatGroq(model=GROQ_MODEL, temperature=0.7)


def hyde_samples(question, n=3):
    docs = []
    for i in range(n):
        text = clean(varied_llm.invoke(HYDE_PROMPT.format(question=question)).content)
        docs.append(text)
        print(f"  sample {i + 1}: {text[:90]}...")
        time.sleep(2.0)          # pace: Groq free tier is ~8000 tokens/minute
    return docs


print("generating 3 hypothetical documents:")
samples = hyde_samples(QUESTION, n=3)

sample_vectors = encoder.encode(samples, normalize_embeddings=True)
mean_vector = sample_vectors.mean(axis=0)
mean_vector /= np.linalg.norm(mean_vector)

multi_scores = doc_vectors @ mean_vector
multi_ids = [int(i) for i in np.argsort(multi_scores)[::-1][:5]]

print("\nmulti-sample HyDE top 5:")
for rank, doc_id in enumerate(multi_ids, 1):
    print(f"  #{rank} doc_{doc_id} (cos={multi_scores[doc_id]:.3f}): "
          f"{paragraphs[doc_id][:95]}...")

generating 3 hypothetical documents:


  sample 1: "Who are you?" said the Caterpillar.

Alice was rather surprised by the question, but she ...


  sample 2: "You are not thinking at all!" said the Caterpillar sharply. "What are you thinking about,...


  sample 3: "You are not the thing you were yesterday," said the Caterpillar, looking up at her with d...



multi-sample HyDE top 5:
  #1 doc_91 (cos=0.745): Which brought them back again to the beginning of the conversation. Alice felt a little irritat...
  #2 doc_93 (cos=0.647): This time Alice waited patiently until it chose to speak again. In a minute or two the Caterpil...
  #3 doc_86 (cos=0.640): “And yet what a dear little puppy it was!” said Alice, as she leant against a buttercup to rest...
  #4 doc_90 (cos=0.637): “Well, perhaps you haven’t found it so yet,” said Alice; “but when you have to turn into a chry...
  #5 doc_20 (cos=0.594): Soon her eye fell on a little glass box that was lying under the table: she opened it, and foun...


### 6. The hybrid trick: keep the question in the mix

A single-vector HyDE query throws the original question away entirely, which is
risky. A common production variant averages the **question vector** with the
hypothetical-document vector, keeping a foot in both worlds.

The weight `alpha` is worth tuning on your own data.

In [9]:
def blended_query(question, hypothetical_doc, alpha=0.5):
    """alpha=0 -> pure question; alpha=1 -> pure HyDE."""
    v = (1 - alpha) * embed(question) + alpha * embed(hypothetical_doc)
    return v / np.linalg.norm(v)


print(f"{'alpha':>6}  top 5 document ids")
print("-" * 46)
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    v = blended_query(QUESTION, hypothetical, alpha)
    ids = [int(i) for i in np.argsort(doc_vectors @ v)[::-1][:5]]
    print(f"{alpha:>6.2f}  {ids}")

 alpha  top 5 document ids
----------------------------------------------
  0.00  [91, 93, 90, 86, 190]
  0.25  [91, 93, 90, 86, 190]
  0.50  [91, 93, 190, 90, 86]
  0.75  [91, 89, 190, 90, 101]
  1.00  [91, 89, 190, 101, 108]


### 7. Full HyDE RAG, end to end

Now the complete loop: hypothesise, retrieve, then generate the *real* answer
strictly from retrieved text. Step 4 is what keeps the hallucinated passage from
reaching the user.

In [10]:
def hyde_rag(question, top_k=3, alpha=1.0):
    hypo = hyde_document(question)
    v = blended_query(question, hypo, alpha)
    ids = [int(i) for i in np.argsort(doc_vectors @ v)[::-1][:top_k]]
    context = "\n\n".join(f"[doc_{i}] {paragraphs[i]}" for i in ids)
    answer = ask(
        "Answer the question using ONLY the context below. Quote the detail you "
        "relied on. If the context does not contain the answer, say so plainly.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    return hypo, ids, answer


hypo, ids, answer = hyde_rag("How does the Cheshire Cat disappear?", top_k=3)

print("hypothetical (discarded after retrieval):")
print(" ", hypo[:220], "...")
print("\nretrieved:", ids)
print("\ngrounded answer:")
print(answer)

hypothetical (discarded after retrieval):
  The Cat grinned at her for a moment, then began to fade away, starting with the end of its tail, and ending with the grin, which remained some time after the rest of it had disappeared. "Well! I had often seen a cat with ...

retrieved: [154, 155, 36]

grounded answer:
The context does not contain the answer.


### 8. Pitfalls

- **Refusals are poison.** If the model says "I do not have enough information",
  you have just embedded a refusal and will retrieve documents about
  uncertainty. Prompt explicitly against hedging, and check for it.
- **HyDE hurts on keyword-ish queries.** Product codes, names, error strings -
  the raw query is already the best possible query. HyDE will blur it. Route
  such queries around the transformation.
- **Domain gap.** If the LLM knows nothing about your private corpus, its
  hypothetical document will be generic and may retrieve generic documents.
  HyDE works best when the model has *stylistic* familiarity even without
  factual knowledge.
- **Never let the hypothetical text reach the user.** It is unsourced model
  output. It exists only to make a vector.
- **Latency doubles.** You now have two sequential LLM calls per query. Cache
  the hypothetical document keyed on the query string.

### 9. Key takeaways

- HyDE embeds a **generated answer** instead of the question, closing the
  question/answer asymmetry gap.
- The hypothetical document's **facts do not matter** - only its shape,
  register, and vocabulary.
- Averaging several samples is cheap variance reduction; blending with the
  question vector (`alpha`) is cheap insurance.
- The final answer must always be generated from **retrieved** text, never from
  the hypothetical passage.

Next: [`03_multi_query_expansion.ipynb`](03_multi_query_expansion.ipynb), which
attacks the same problem from a different angle - many phrasings instead of one
better phrasing.